# Fraud Detection Pipeline
## Chapter 17 — End-to-End ML System

This notebook:
1. Connects to `shop.db` (operational database)
2. Builds an analytical warehouse table via ETL
3. Trains a fraud classification model
4. Saves the model artifact (`model.pkl`)
5. Evaluates performance

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt

DB_PATH = Path("../data/shop.db")
MODEL_PATH = Path("../jobs/model.pkl")
print(f"Database: {DB_PATH.resolve()}")

## 1 — Extract: Load raw tables

In [ ]:
conn = sqlite3.connect(DB_PATH)

customers = pd.read_sql("SELECT * FROM customers", conn)
orders = pd.read_sql("SELECT * FROM orders", conn)
order_items = pd.read_sql("SELECT * FROM order_items", conn)
shipments = pd.read_sql("SELECT * FROM shipments", conn)

print(f"customers:  {customers.shape}")
print(f"orders:     {orders.shape}")
print(f"order_items:{order_items.shape}")
print(f"shipments:  {shipments.shape}")

## 2 — Transform: Build warehouse table

We join orders with customer demographics, aggregate order-item stats, and engineer features useful for fraud detection.

In [ ]:
# Aggregate order_items per order
item_agg = (
    order_items.groupby("order_id")
    .agg(
        num_items=("quantity", "sum"),
        num_distinct_products=("product_id", "nunique"),
        avg_unit_price=("unit_price", "mean"),
        max_unit_price=("unit_price", "max"),
    )
    .reset_index()
)

item_agg.head()

In [ ]:
# Join everything into one analytical table
warehouse = (
    orders
    .merge(customers[["customer_id", "gender", "customer_segment", "loyalty_tier", "state"]],
           on="customer_id", how="left", suffixes=("", "_cust"))
    .merge(item_agg, on="order_id", how="left")
    .merge(shipments[["order_id", "carrier", "shipping_method", "distance_band"]],
           on="order_id", how="left")
)

# Feature engineering
warehouse["zip_mismatch"] = (warehouse["billing_zip"] != warehouse["shipping_zip"]).astype(int)
warehouse["is_foreign_ip"] = (warehouse["ip_country"] != "US").astype(int)
warehouse["order_hour"] = pd.to_datetime(warehouse["order_datetime"]).dt.hour

print(f"Warehouse table: {warehouse.shape}")
warehouse.head()

### Load: Write warehouse table back to database

In [ ]:
warehouse.to_sql("warehouse_fraud", conn, if_exists="replace", index=False)
print("Wrote warehouse_fraud table to shop.db")

## 3 — Model Training

In [ ]:
# Define features
categorical_cols = [
    "payment_method", "device_type", "ip_country",
    "gender", "customer_segment", "loyalty_tier",
    "carrier", "shipping_method", "distance_band",
]

numeric_cols = [
    "order_subtotal", "shipping_fee", "tax_amount", "order_total",
    "risk_score", "promo_used",
    "num_items", "num_distinct_products", "avg_unit_price", "max_unit_price",
    "zip_mismatch", "is_foreign_ip", "order_hour",
]

target = "is_fraud"

X = warehouse[categorical_cols + numeric_cols].copy()
y = warehouse[target].copy()

print(f"Features: {X.shape[1]}  |  Fraud rate: {y.mean():.1%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Train fraud rate: {y_train.mean():.1%}  |  Test fraud rate: {y_test.mean():.1%}")

In [ ]:
# Build sklearn pipeline with preprocessing + model
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

clf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

clf.fit(X_train, y_train)
print("Model trained.")

## 4 — Evaluation

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Legit", "Fraud"], ax=axes[0], cmap="Blues"
)
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (top 15)
ohe_names = clf.named_steps["preprocessor"].transformers_[0][1].get_feature_names_out(categorical_cols)
all_features = list(ohe_names) + numeric_cols
importances = clf.named_steps["classifier"].feature_importances_

feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False).head(15)

feat_imp.plot.barh(figsize=(8, 5), title="Top 15 Feature Importances")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5 — Save Model Artifact

In [ ]:
with open(MODEL_PATH, "wb") as f:
    pickle.dump({
        "pipeline": clf,
        "categorical_cols": categorical_cols,
        "numeric_cols": numeric_cols,
    }, f)

print(f"Model saved to {MODEL_PATH.resolve()}")

In [ ]:
conn.close()
print("Done. Run jobs/run_inference.py to score new orders.")